In [2]:
"""Utilities for the Deep-SVDD continual-learning benchmark.

Contents
  1. Model + data
  2. Losses / parameter helpers / spectral helpers
  3. Hessian (stiff/sloppy) subspace
  4. Baselines: online EWC (GN Fisher), Jacobian-OGD, per-layer GPM, per-layer OWM
  5. Training of one sequential task
  6. Result logging, summaries, forgetting table, quadrant plot
"""
import os
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.sparse.linalg import LinearOperator, eigsh
from torch.func import functional_call, grad, jacrev, jvp, vmap
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST

REP_DIM = 4
MAX_TASKS = 4  # longest task sequence; sets the number of dist_t{i} columns
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081


# =============================================================================
# 1. Model + data
# =============================================================================
class SVDDNetwork(nn.Module):

  def __init__(self, hidden_sizes=(10, 8), out_dim=4):
    super().__init__()
    layers = []
    in_size = 28 * 28
    for h in hidden_sizes:
      layers += [nn.Linear(in_size, h), nn.ReLU()]
      in_size = h
    layers.append(nn.Linear(in_size, out_dim, bias=False))
    self.net = nn.Sequential(*layers)

  def forward(self, x):
    return self.net(x.view(x.size(0), -1))


def load_mnist_digit_tensors(root="./data", val_frac=0.1, split_seed=0):
  """Returns {digit: {"train","val","test"}} of normalized (N,1,28,28) tensors.

  Built once, vectorized (no per-image transform loop). The train/val split is fixed
  by `split_seed` so every method/seed sees the same validation images.
  """
  train_raw = MNIST(root=root, train=True, download=True)
  test_raw = MNIST(root=root, train=False, download=True)

  def norm(d):
    return ((d.float() / 255.0 - MNIST_MEAN) / MNIST_STD).unsqueeze(1)

  g = torch.Generator().manual_seed(split_seed)
  out = {}
  for digit in range(10):
    xtr = norm(train_raw.data[train_raw.targets == digit])
    perm = torch.randperm(len(xtr), generator=g)
    n_val = int(len(xtr) * val_frac)
    out[digit] = {
        "train": xtr[perm[n_val:]],
        "val": xtr[perm[:n_val]],
        "test": norm(test_raw.data[test_raw.targets == digit]),
    }
  return out


def make_loader(x, batch_size, shuffle):
  ds = TensorDataset(x, torch.zeros(len(x), dtype=torch.long))
  return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def sample_memory(x, n=1024):
  return x[torch.randperm(len(x))[:n]]


# =============================================================================
# 2. Losses / parameter helpers / spectral helpers
# =============================================================================
def svdd_loss(outputs, center):
  return torch.mean(torch.sum((outputs - center) ** 2, dim=1))


def get_param_vec(model):
  return torch.cat([p.reshape(-1) for p in model.parameters()])


def flatten_params_dict(param_dict):
  return torch.cat([p.reshape(-1) for p in param_dict.values()])


def evaluate_dist(model, loader, center, device):
  model.eval()
  total_dist, count = 0.0, 0
  with torch.no_grad():
    for x, _ in loader:
      x = x.to(device)
      out = model(x)
      total_dist += torch.norm(out - center, dim=1).sum().item()
      count += x.size(0)
  return total_dist / count


def find_spectral_variance_k(evals, threshold=0.90, min_k=1, max_k=100):
  pos_evals = np.maximum(evals[:max_k], 0.0)
  total_energy = pos_evals.sum()
  if total_energy <= 0:
    return min_k
  cumsum = np.cumsum(pos_evals) / total_energy
  k = int(np.searchsorted(cumsum, threshold)) + 1
  return max(min_k, k)


def _linears(model):
  return [m for m in model.modules() if isinstance(m, nn.Linear)]


def _detached_params(model):
  return {k: v.detach() for k, v in model.named_parameters()}


def _jacobian_fn(model):
  """Per-sample Jacobian of the network output w.r.t. all parameters."""

  def f(p, xi):
    return functional_call(model, p, (xi.unsqueeze(0),)).squeeze(0)

  return vmap(jacrev(f), in_dims=(None, 0))


# =============================================================================
# 3. Hessian (stiff / sloppy) subspace
# =============================================================================
def compute_hessian_subspace(model, batches, device, k_max=100):
  """Top-k Hessian eigenvectors of the mean SVDD loss over `batches` [(x, center), ...].

  Returns (subspace (k, P), k_elbow, theta_star_dict).
  torch.compile was dropped: the function was rebuilt on every call, so it recompiled each time.
  """
  model.eval()
  theta_star_dict = {k: v.detach().clone() for k, v in model.named_parameters()}
  params_at_theta = OrderedDict(
      (k, v.to(device).clone()) for k, v in theta_star_dict.items()
  )

  param_keys = list(params_at_theta.keys())
  param_shapes = [v.shape for v in params_at_theta.values()]
  param_numels = [v.numel() for v in params_at_theta.values()]
  slices, start = [], 0
  for n in param_numels:
    slices.append(slice(start, start + n))
    start += n
  num_params = start

  def functional_loss(params, data, c):
    return svdd_loss(functional_call(model, params, (data,)), c)

  grad_func = grad(functional_loss)

  def compute_batch_jvp(tangents_dict, params_dict, data, c):
    zeros_data = torch.zeros_like(data)
    zeros_c = torch.zeros_like(c)
    return jvp(
        grad_func, (params_dict, data, c), (tangents_dict, zeros_data, zeros_c)
    )[1]

  batched_jvp_op = vmap(compute_batch_jvp, in_dims=(0, None, None, None))

  def hvp_block_wrapper(V_np):
    if V_np.ndim == 1:
      V_np = V_np[:, None]
    V_tensor = torch.from_numpy(V_np).to(device).float().T.contiguous()
    bs = V_tensor.shape[0]
    v_dict_batch = OrderedDict(
        (k, V_tensor[:, s].reshape(bs, *shp))
        for k, s, shp in zip(param_keys, slices, param_shapes)
    )
    acc = None
    for x_batch, center_i in batches:
      out_dict = batched_jvp_op(v_dict_batch, params_at_theta, x_batch, center_i)
      out_flat = torch.cat(
          [out_dict[k].reshape(bs, -1) for k in param_keys], dim=1
      )
      acc = out_flat if acc is None else acc + out_flat
    avg = acc / float(len(batches))
    out_np = avg.detach().cpu().numpy().astype(np.float32)
    return np.nan_to_num(out_np, nan=0.0, posinf=0.0, neginf=0.0)

  lin_op = LinearOperator(
      (num_params, num_params),
      matvec=lambda v: hvp_block_wrapper(v).flatten(),
      matmat=hvp_block_wrapper,
      dtype=np.float32,
  )

  safe_k = min(k_max, num_params - 1)
  rng = np.random.default_rng(42)
  v0 = rng.standard_normal(num_params).astype(np.float32)

  try:
    vals, vecs = eigsh(
        lin_op, k=safe_k, which="LA", v0=v0, tol=1e-3, maxiter=num_params * 10
    )
  except Exception as e:  # pylint: disable=broad-except
    print(f"eigsh failed with error: {e}. Falling back to zero subspace.")
    vals = np.zeros(safe_k, dtype=np.float32)
    vecs = np.zeros((num_params, safe_k), dtype=np.float32)

  idx = np.argsort(vals)[::-1]
  vals, vecs = vals[idx], vecs[:, idx]
  pos_mask = vals > 1e-8
  pos_vals, pos_vecs = vals[pos_mask], vecs[:, pos_mask]

  if pos_vecs.shape[1] == 0:
    subspace = torch.zeros((1, num_params), device=device)
    k_elbow = 1
  else:
    subspace = torch.from_numpy(pos_vecs.copy()).float().to(device).T
    k_elbow = find_spectral_variance_k(pos_vals, threshold=0.90, max_k=k_max)

  return subspace, k_elbow, theta_star_dict


# =============================================================================
# 4. Baselines
# =============================================================================
@torch.no_grad()
def collect_layer_inputs(model, batches, device, max_per_batch=256):
  """Input activations of every Linear layer, list of (in_dim, N) float64 tensors."""
  model.eval()
  linears = _linears(model)
  store = [[] for _ in linears]
  hooks = []
  for idx, layer in enumerate(linears):

    def hook(mod, inp, out, idx=idx):
      store[idx].append(inp[0].detach()[:max_per_batch])

    hooks.append(layer.register_forward_hook(hook))
  try:
    for x_batch, _ in batches:
      model(x_batch.to(device))
  finally:
    for h in hooks:
      h.remove()
  return [torch.cat(s, dim=0).double().T.contiguous() for s in store]


# ---- EWC: online accumulation + Gauss-Newton diagonal Fisher -----------------
def compute_gn_fisher(model, loader, device):
  """2 * mean_x sum_k (df_k/dtheta)^2: diag Gauss-Newton Hessian of the mean SVDD loss.

  With this scaling ewc_lambda = 1 matches the second-order approximation of the loss increase.
  """
  model.eval()
  params = _detached_params(model)
  jac_fn = _jacobian_fn(model)
  fisher = {k: torch.zeros_like(v) for k, v in params.items()}
  n = 0
  for x, _ in loader:
    J = jac_fn(params, x.to(device))  # each: (B, out_dim, *param_shape)
    for k in fisher:
      fisher[k] += 2.0 * (J[k] ** 2).sum(dim=(0, 1))
    n += x.size(0)
  return {k: v / n for k, v in fisher.items()}


def merge_ewc(F_old, theta_old, F_new, theta_new, eps=1e-12):
  """Merge quadratic penalties: sum_t F_t (th - th_t)^2 = F (th - th_eff)^2 + const."""
  if F_old is None:
    return F_new, {k: v.detach().clone() for k, v in theta_new.items()}
  F, th = {}, {}
  for k in F_new:
    F[k] = F_old[k] + F_new[k]
    merged = (F_old[k] * theta_old[k] + F_new[k] * theta_new[k].detach()) / (F[k] + eps)
    th[k] = torch.where(F[k] > eps, merged, theta_new[k].detach())
  return F, th


def ewc_proximal_step(model, fisher, theta_star, lr, ewc_lambda):
  """Implicit (proximal) step for the EWC penalty (lambda/2) F (theta - theta*)^2.

  Explicit SGD on the penalty diverges when lr * lambda * F > 2, which happens for large lambda
  once Fishers accumulate. The proximal step has the same stationary point and is unconditionally stable.
  """
  with torch.no_grad():
    for name, p in model.named_parameters():
      th = theta_star[name]
      p.copy_(th + (p - th) / (1.0 + lr * ewc_lambda * fisher[name]))


# ---- OGD: output-Jacobian rows, incremental SVD basis ------------------------
def jacobian_rows(model, x):
  model.eval()
  params = _detached_params(model)
  J = _jacobian_fn(model)(params, x)
  B = x.size(0)
  rows = torch.cat([J[k].reshape(B, J[k].shape[1], -1) for k in params], dim=2)
  return rows.reshape(-1, rows.shape[2])  # (B * out_dim, num_params)


def update_ogd_basis(model, batches, device, basis=None, n_samples=200, tol=1e-2):
  """Extends the orthonormal basis (k, P) with the new task's output-Jacobian directions."""
  x = torch.cat([xb for xb, _ in batches], dim=0)[:n_samples].to(device)
  rows = jacobian_rows(model, x).double()
  ref = rows.norm(dim=1).max()
  if basis is not None:
    b = basis.double()
    rows = rows - (rows @ b.T) @ b
  _, S, Vh = torch.linalg.svd(rows, full_matrices=False)
  new = Vh[S > tol * ref].float()
  if basis is None:
    return new if new.shape[0] > 0 else None
  return torch.cat([basis, new], dim=0) if new.shape[0] > 0 else basis


# ---- GPM: per-layer activation subspaces, incremental residual SVD -----------

@torch.no_grad()
def update_gpm_bases(model, batches, device, bases=None, threshold=0.97, max_frac=0.9):
  acts = collect_layer_inputs(model, batches, device)
  thr = list(threshold) if isinstance(threshold, (list, tuple)) else [threshold] * len(acts)
  out = []
  for l, X in enumerate(acts):
    cap = max(1, int(max_frac * X.shape[0]))      # keep >= (1-max_frac) of dims free
    total = (X ** 2).sum()
    M = None if bases is None else bases[l]
    if M is None:
      U, S, _ = torch.linalg.svd(X, full_matrices=False)
      ratio = torch.cumsum(S ** 2, 0) / total
      k = int(torch.searchsorted(ratio, ratio.new_tensor([thr[l]])).item()) + 1
      out.append(U[:, : min(k, U.shape[1], cap)])
    else:
      room = cap - M.shape[1]
      proj = M.T @ X
      proj_e = (proj ** 2).sum()
      if proj_e / total >= thr[l] or room <= 0:
        out.append(M)
        continue
      U, S, _ = torch.linalg.svd(X - M @ proj, full_matrices=False)
      ratio = (proj_e + torch.cumsum(S ** 2, 0)) / total
      k = int(torch.searchsorted(ratio, ratio.new_tensor([thr[l]])).item()) + 1
      out.append(torch.cat([M, U[:, : min(k, U.shape[1], room)]], dim=1))
  return out

def gpm_projectors(bases):
  return [
      torch.eye(M.shape[0], dtype=M.dtype, device=M.device) - M @ M.T
      for M in bases
  ]



@torch.no_grad()
def update_owm_projectors(model, batches, device, Ps=None, alpha_rel=1e-2, max_per_batch=128):
  """alpha = alpha_rel * lambda_max(A A^T) per layer: directions with energy >> alpha_rel
  of the top direction are protected, weaker ones stay free (a soft rank cutoff)."""
  acts = collect_layer_inputs(model, batches, device, max_per_batch)
  new = []
  for l, A in enumerate(acts):
    in_dim, N = A.shape
    P = torch.eye(in_dim, dtype=A.dtype, device=A.device) if Ps is None else Ps[l]
    alpha = alpha_rel * torch.linalg.matrix_norm(A, ord=2) ** 2   # scale from raw A, not P@A
    PA = P @ A
    inner = alpha * torch.eye(N, dtype=A.dtype, device=A.device) + A.T @ PA
    new.append(P - PA @ torch.linalg.solve(inner, PA.T))
  return new


def project_layer_grads(model, Ps):
  """GPM/OWM: weight gradient G -> G @ P per layer. Biases are left free."""
  for lin, P in zip(_linears(model), Ps):
    if lin.weight.grad is not None:
      lin.weight.grad = lin.weight.grad @ P.to(lin.weight.grad.dtype)


# ---- Hessian-subspace / OGD projected SGD ------------------------------------
def projected_sgd_step(model, lr, basis, mode):
  """sloppy / ogd: remove the basis component. stiff: keep only the basis component."""
  g = torch.cat([p.grad.reshape(-1) for p in model.parameters()])
  if basis is None:
    g_proj = g
  else:
    g_in = basis.T @ (basis @ g)
    g_proj = g_in if mode == "stiff" else g - g_in
  idx = 0
  with torch.no_grad():
    for p in model.parameters():
      n = p.numel()
      p -= lr * g_proj[idx: idx + n].view_as(p)
      idx += n

@torch.no_grad()
def max_output_drift(model_a, model_b, batches, device, n=128):
  """Max |f_a(x) - f_b(x)| on the first n stored samples per task (the ones the bases were built from)."""
  model_a.eval(); model_b.eval()
  worst = 0.0
  for x, _ in batches:
    x = x[:n].to(device)
    worst = max(worst, (model_a(x) - model_b(x)).abs().max().item())
  return worst
# =============================================================================
# 5. Training of one sequential task
# =============================================================================
def train_sequential_task(
    model_state,
    theta_star_dict,
    constraint_basis,
    fisher_dict,
    past_tasks,
    cur_task,
    train_loader,
    k,
    constraint_type,
    device,
    ewc_lambda=1.0,
    epochs=20,
    lr=0.01,
    freeze_bias=True
):
  """Trains one task. Returns (model, metrics).

  constraint_type / constraint_basis:
    none            -> None
    ewc             -> fisher_dict + theta_star_dict (merged online EWC), ewc_lambda
    sloppy / stiff  -> (K, P) Hessian eigenvectors (first k rows used), theta_star_dict for angles
    ogd             -> (K, P) orthonormal basis, or None
    gpm / owm       -> list of per-layer (in, in) projectors (use gpm_projectors for GPM)
  past_tasks / cur_task: dicts with "center", "val_loader", "test_loader".
  """
  model = SVDDNetwork(out_dim=REP_DIM).to(device)
  model.load_state_dict(model_state)
  center = cur_task["center"]

  use_subspace = constraint_type in ("stiff", "sloppy")
  effective_basis = constraint_basis
  if use_subspace and constraint_basis is not None:
    effective_basis = constraint_basis[: min(k, constraint_basis.shape[0])]
  theta_star_vec = (
      flatten_params_dict(theta_star_dict).detach()
      if use_subspace and theta_star_dict is not None
      else None
  )

  angles = []
  for _ in range(epochs):
    model.train()
    for x, _ in train_loader:
      x = x.to(device)
      loss = svdd_loss(model(x), center)
      model.zero_grad()
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

      if constraint_type in ("none", "ewc", "gpm", "owm"):
        if constraint_type in ("gpm", "owm") and constraint_basis is not None:
          project_layer_grads(model, constraint_basis)
          if freeze_bias:
            for lin in _linears(model):
              if lin.bias is not None:
                lin.bias.grad.zero_()
        with torch.no_grad():
          for p in model.parameters():
            p -= lr * p.grad
        if constraint_type == "ewc" and fisher_dict is not None:
          ewc_proximal_step(model, fisher_dict, theta_star_dict, lr, ewc_lambda)
      else:  # sloppy, stiff, ogd
        projected_sgd_step(model, lr, effective_basis, constraint_type)
        if use_subspace and effective_basis is not None and theta_star_vec is not None:
          with torch.no_grad():
            delta = get_param_vec(model) - theta_star_vec
            if torch.norm(delta) > 1e-9:
              proj = (delta @ effective_basis.T) @ effective_basis
              cosine = torch.clamp(torch.norm(proj) / torch.norm(delta), 0.0, 1.0)
              angles.append((torch.acos(cosine) * 57.2958).item())

  past_test = [
      evaluate_dist(model, t["test_loader"], t["center"], device) for t in past_tasks
  ]
  curr_test = evaluate_dist(model, cur_task["test_loader"], center, device)
  val_dists = [
      evaluate_dist(model, t["val_loader"], t["center"], device)
      for t in list(past_tasks) + [cur_task]
  ]
  metrics = {
      "past_avg_dist": float(np.mean(past_test)) if past_test else 0.0,
      "curr_dist": curr_test,
      "avg_angle": float(np.mean(angles)) if angles else 0.0,
      # model selection criterion: mean VALIDATION distance over all tasks seen so far
      "val_score": float(np.mean(val_dists)),
      "task_dists": past_test + [curr_test],
  }
  return model, metrics


# =============================================================================
# 6. Logging, summaries, plots
# =============================================================================
def make_row(seq_idx, seed, step, digit, method, config, k, selected, metrics):
  row = {
      "SEQUENCE_IDX": seq_idx,
      "SEED": seed,
      "STEP": step + 1,
      "TASK_DIGIT": digit,
      "method": method,
      "config": config,
      "k": k,
      "selected": bool(selected),
      "past_avg_dist": metrics["past_avg_dist"],
      "curr_dist": metrics["curr_dist"],
      "avg_angle": metrics["avg_angle"],
      "val_score": metrics["val_score"],
  }
  td = metrics["task_dists"]
  for i in range(MAX_TASKS):
    row[f"dist_t{i}"] = td[i] if i < len(td) else np.nan
  return row


def summarize_results(raw_csv, summary_csv):
  """Seed-averaged table of the SELECTED config per (sequence, step, method)."""
  df = pd.read_csv(raw_csv)
  sel = df[df["selected"]]
  summary = sel.groupby(
      ["SEQUENCE_IDX", "STEP", "TASK_DIGIT", "method"], as_index=False
  ).agg(
      past_avg_dist=("past_avg_dist", "mean"),
      past_avg_dist_std=("past_avg_dist", "std"),
      curr_dist=("curr_dist", "mean"),
      curr_dist_std=("curr_dist", "std"),
      avg_angle=("avg_angle", "mean"),
      k_mean=("k", "mean"),
      n_seeds=("SEED", "nunique"),
  )
  summary.to_csv(summary_csv, index=False)
  return summary



def plot_aggregated_benchmark_quadrant(raw_csv, save_filename="aggregated_quadrant_benchmark.png"):
  """Final-step scatter: x = distance on past tasks (forgetting), y = distance on the newest task
  (plasticity). Both lower-is-better; dashed lines mark the unconstrained `none` baseline, so the
  bottom-left quadrant is better than `none` on both axes. Error bars: std over sequences x seeds."""
  import matplotlib
  matplotlib.use("Agg")
  import matplotlib.pyplot as plt

  df = pd.read_csv(raw_csv)
  final = df[df["selected"] & (df["STEP"] == df["STEP"].max())]
  agg = final.groupby("method").agg(
      px=("past_avg_dist", "mean"), px_sd=("past_avg_dist", "std"),
      cy=("curr_dist", "mean"), cy_sd=("curr_dist", "std"),
  ).fillna(0.0)

  fig, ax = plt.subplots(figsize=(7.5, 6))
  for method, r in agg.iterrows():
    ax.errorbar(r["px"], r["cy"], xerr=r["px_sd"], yerr=r["cy_sd"], fmt="o", capsize=3, label=method)
  if "none" in agg.index:
    ax.axvline(agg.loc["none", "px"], ls="--", c="gray", lw=0.8)
    ax.axhline(agg.loc["none", "cy"], ls="--", c="gray", lw=0.8)
  ax.set_xlabel("Past-task distance to center (lower = less forgetting)")
  ax.set_ylabel("Newest-task distance to center (lower = more plastic)")
  ax.set_title("Stability vs plasticity at final step")
  ax.legend()
  fig.tight_layout()
  fig.savefig(save_filename, dpi=150)
  plt.close(fig)


c:\Users\harip\anaconda3\Lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [ ]:
import copy
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.optim as optim


torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TASK_SEQUENCES = [[1, 2, 3, 4], [5, 6, 7, 8], [0, 2, 4, 6], [1, 3, 5,7], [4,5,6,7], [2,3,4,5]]
#TASK_SEQUENCES = [[1, 2, 3, 4]]
SEEDS = [42, 43, 44]
METHODS = ["none", "ewc", "sloppy", "stiff", "ogd", "gpm", "owm"]

# With the Gauss-Newton Fisher, lambda = 1 is the "principled" strength.
EWC_LAMBDAS = [0.1, 1, 10, 100, 1000]
K_FACTORS = [0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]
GPM_THRESHOLD = 0.97
OWM_ALPHA = 1e-3
GPM_MAX_FRAC = 0.9
OWM_ALPHA_REL = 1e-2

RAW_CSV = "raw_results_benchmark_mnist_5sequences.csv"          # every config, every seed
SUMMARY_CSV = "detailed_results_benchmark_mnist_5sequences.csv"  # selected config, seed-averaged
PLOT_FILE = "aggregated_quadrant_benchmark_mnist_5sequences.png"
FREEZE_BIAS = True
assert max(len(s) for s in TASK_SEQUENCES) <= MAX_TASKS


def one_hot_center(step):
  c = torch.zeros(REP_DIM, device=DEVICE)
  c[step % REP_DIM] = 1.0
  return c


def run_one(seq_idx, sequence, seed, digit_data):
  """Runs all methods on one (sequence, seed). Returns a list of result rows."""
  torch.manual_seed(seed)
  np.random.seed(seed)
  rows = []

  base_model = SVDDNetwork(out_dim=REP_DIM).to(DEVICE)
  models = {}                 # method -> its own trajectory's model
  past_tasks = []             # shared (same tasks for every method)
  memory_batches = []         # shared [(x, center)] used by the Hessian methods

  ogd_basis, gpm_bases, owm_Ps = None, None, None
  ewc_F, ewc_theta = None, None
  sub, kel, theta = {}, {}, {}  # Hessian state for "sloppy"/"stiff"

  for step, digit in enumerate(sequence):
    center = one_hot_center(step)
    d = digit_data[digit]
    loader_train = make_loader(d["train"], 1024, True)
    task = {
        "digit": digit,
        "center": center,
        "val_loader": make_loader(d["val"], 1000, False),
        "test_loader": make_loader(d["test"], 1000, False),
    }
    base = dict(seq_idx=seq_idx, seed=seed, step=step, digit=digit)
    print(f"  --- step {step + 1}: digit {digit} ---")

    if step == 0:
      opt = optim.Adam(base_model.parameters(), lr=1e-3, weight_decay=1e-5)
      for _ in range(10):
        base_model.train()
        for x, _ in loader_train:
          x = x.to(DEVICE)
          opt.zero_grad()
          svdd_loss(base_model(x), center).backward()
          opt.step()

      curr = evaluate_dist(base_model, task["test_loader"], center, DEVICE)
      val = evaluate_dist(base_model, task["val_loader"], center, DEVICE)
      metrics = dict(past_avg_dist=0.0, curr_dist=curr, avg_angle=0.0,
                     val_score=val, task_dists=[curr])
      for m in METHODS:
        models[m] = copy.deepcopy(base_model)
        rows.append(make_row(**base, method=m, config="-", k=-1, selected=True, metrics=metrics))

    else:
      
      def run(ctype, model, theta_d=None, basis=None, fisher=None, k=0, lam=1.0):
        return train_sequential_task(
            model.state_dict(), theta_d, basis, fisher, past_tasks, task,
            loader_train, k, ctype, DEVICE, ewc_lambda=lam, freeze_bias=FREEZE_BIAS)

      def record(method, candidates):
        """Logs all candidates, selects by validation score, returns the selected model."""
        best = int(np.argmin([c["metrics"]["val_score"] for c in candidates]))
        for i, c in enumerate(candidates):
          rows.append(make_row(**base, method=method, config=c["config"], k=c["k"],
                               selected=(i == best), metrics=c["metrics"]))
        return candidates[best]["model"]

      # 1. unconstrained SGD
      model, met = run("none", models["none"])
      models["none"] = record("none", [dict(config="-", k=-1, model=model, metrics=met)])

      # 2. EWC: sweep lambda on EWC's own trajectory
      cands = []
      for lam in EWC_LAMBDAS:
        model, met = run("ewc", models["ewc"], theta_d=ewc_theta, fisher=ewc_F, lam=lam)
        cands.append(dict(config=f"lam={lam}", k=-1, model=model, metrics=met))
      models["ewc"] = record("ewc", cands)

      # 3. sloppy / stiff Hessian subspaces: sweep k around the spectral elbow
      for sc in ("sloppy", "stiff"):
        k_to_factor = {}
        for f in K_FACTORS:
          kk = max(1, min(int(round(kel[sc] * f)), sub[sc].shape[0]))
          k_to_factor.setdefault(kk, f)
        cands = []
        for kk, f in sorted(k_to_factor.items()):
          model, met = run(sc, models[sc], theta_d=theta[sc], basis=sub[sc], k=kk)
          cands.append(dict(config=f"x{f}", k=kk, model=model, metrics=met))
        models[sc] = record(sc, cands)

      # 4. OGD
      model, met = run("ogd", models["ogd"], basis=ogd_basis)
      models["ogd"] = record("ogd", [dict(config="-", k=-1, model=model, metrics=met)])

      # 5. GPM
      old = models["gpm"]
      model, met = run("gpm", models["gpm"], basis=gpm_projectors(gpm_bases))
      print(f"    gpm drift on stored samples: {max_output_drift(old, model, memory_batches, DEVICE):.2e}")
      models["gpm"] = record("gpm", [dict(config=f"thr={GPM_THRESHOLD}", k=-1, model=model, metrics=met)])

      # 6. OWM
      model, met = run("owm", models["owm"], basis=owm_Ps)
      models["owm"] = record("owm", [dict(config=f"alpha={OWM_ALPHA}", k=-1, model=model, metrics=met)])

    # ---- memory update after learning this task (skipped after the last one) ----
    past_tasks.append(task)
    x_mem = sample_memory(d["train"]).to(DEVICE)
    memory_batches.append((x_mem, center))
    if step < len(sequence) - 1:
      newest = [memory_batches[-1]]  # GPM / OWM / OGD take only the newest task
      ogd_basis = update_ogd_basis(models["ogd"], newest, DEVICE, ogd_basis)
      gpm_bases = update_gpm_bases(models["gpm"], newest, DEVICE, gpm_bases,
                             threshold=GPM_THRESHOLD, max_frac=GPM_MAX_FRAC)
      owm_Ps = update_owm_projectors(models["owm"], newest, DEVICE, owm_Ps, alpha_rel=OWM_ALPHA_REL)
      print("    free dims  GPM:", [M.shape[0] - M.shape[1] for M in gpm_bases],
            " OWM:", [round(torch.trace(P).item(), 1) for P in owm_Ps],
            " OGD rank:", None if ogd_basis is None else ogd_basis.shape[0])

      F_new = compute_gn_fisher(models["ewc"], loader_train, DEVICE)
      ewc_F, ewc_theta = merge_ewc(
          ewc_F, ewc_theta, F_new,
          {k: v.detach() for k, v in models["ewc"].named_parameters()})

      for sc in ("sloppy", "stiff"):  # Hessian only where it is used
        sub[sc], kel[sc], theta[sc] = compute_hessian_subspace(
            models[sc], memory_batches, DEVICE, k_max=100)
      print(f"    k_elbow  sloppy={kel['sloppy']}  stiff={kel['stiff']}")

  return rows


def main():
  for path in (RAW_CSV, SUMMARY_CSV):
    if os.path.exists(path):
      os.remove(path)

  digit_data = load_mnist_digit_tensors("./data")

  for seq_idx, sequence in enumerate(TASK_SEQUENCES):
    for seed in SEEDS:
      print(f"\n=== Sequence {seq_idx + 1}: {sequence} | seed {seed} ===")
      t0 = time.time()
      rows = run_one(seq_idx, sequence, seed, digit_data)
      pd.DataFrame(rows).to_csv(
          RAW_CSV, mode="a", header=not os.path.exists(RAW_CSV), index=False)
      print(f"  done in {time.time() - t0:.0f}s")

  summarize_results(RAW_CSV, SUMMARY_CSV)
  plot_aggregated_benchmark_quadrant(RAW_CSV, save_filename=PLOT_FILE)
  print(f"\nSaved {RAW_CSV}, {SUMMARY_CSV}, {PLOT_FILE}")


if __name__ == "__main__":
  main()


In [1]:
import pandas as pd
RAW_CSV = "raw_results_benchmark_mnist_5sequences.csv"  
df = pd.read_csv(RAW_CSV)
df = df.drop(columns=["avg_angle", "val_score"])
def color_rule(val):
    if val < 1:
        return "background-color: #90EE90"  # green
    elif val < 2:
        return "background-color: #FFD580"  # yellow
    else:
        return "background-color: #FF7F7F"  # red

styled = df.style.applymap(color_rule, subset=df.columns[[8,9,10,11,12,13]])
styled

In [2]:
latex = styled.to_latex(
    convert_css=True,
    hrules=True,
    environment="longtable",
)

with open("table_mnist.tex", "w") as f:
    f.write(latex)